In [1]:
import os, subprocess
import json
import uproot3
import awkward as ak
import numpy as np
from coffea import processor, util, hist
import pickle

from plotter import *

In [2]:
lumis = {}
lumis['2016'] = 35.9
lumis['2017'] = 41.5
lumis['2018'] = 59.9

with open('xsec.json') as f:
  xs = json.load(f)

with open('pmap.json') as f:
  pmap = json.load(f)

systematics = ['nominal',
               'jet_triggerUp','jet_triggerDown',
               'btagWeightUp','btagWeightDown','btagEffStatUp','btagEffStatDown',
               'UESUp','UESDown','JESUp','JESDown','JERUp','JERDown',
               'pileup_weightUp','pileup_weightDown'
              ]

In [3]:
ddbthr = 0.7

In [4]:
year = '2017'
nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
outsum = processor.dict_accumulator()

In [5]:
repickle=True

# Check if pickle exists, and don't re-create it if it does
picklename = 'pickles/'+str(year)+'_templates.pkl'
if os.path.isfile(picklename):
    repickle=False

In [6]:
# Load all files - this takes a while
if repickle:
    nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
    for n in range(1,nfiles+1):

        with open('infiles-split/'+year+'_'+str(n)+'.json') as f:
            infiles = json.load(f)
    
        filename = '/myeosdir/ggf-vbf/outfiles-ddb2/'+year+'_'+str(n)+'.coffea'
        if os.path.isfile(filename):
            out = util.load(filename)
            outsum.add(out)
        else:
            print('Missing file '+str(n),infiles.keys())
            #print("File " + filename + " is missing")
        
    scale_lumi = {k: xs[k] * 1000 *lumis[year] / w for k, w in outsum['sumw'].items()} 
    outsum['templates'].scale(scale_lumi, 'dataset')
    print(outsum['templates'].identifiers('dataset'))
    templates = outsum['templates'].group('dataset', hist.Cat('process', 'Process'), pmap)
          
    outfile = open(picklename, 'wb')
    pickle.dump(templates, outfile, protocol=-1)
    outfile.close()

Missing file 2416 dict_keys(['WJetsToLNu_HT-800To1200_TuneCP5_13TeV-madgraphMLM-pythia8'])
[<StringBin (DYJetsToLL_M-50_HT-1200to2500_TuneCP5_13TeV-madgraphMLM-pythia8) instance at 0x7f757d503a90>, <StringBin (DYJetsToLL_M-50_HT-2500toInf_TuneCP5_13TeV-madgraphMLM-pythia8) instance at 0x7f757d508940>, <StringBin (DYJetsToLL_M-50_HT-400to600_TuneCP5_13TeV-madgraphMLM-pythia8) instance at 0x7f757d508ee0>, <StringBin (DYJetsToLL_M-50_HT-600to800_TuneCP5_13TeV-madgraphMLM-pythia8) instance at 0x7f757d508880>, <StringBin (DYJetsToLL_M-50_HT-800to1200_TuneCP5_13TeV-madgraphMLM-pythia8) instance at 0x7f75d1dea1c0>, <StringBin (GluGluHToBB_M-125_13TeV_powheg_MINLO_NNLOPS_pythia8) instance at 0x7f75d1df7430>, <StringBin (GluGluHToBB_M125_TuneCP5_13TeV-powheg-pythia8) instance at 0x7f75d1d44df0>, <StringBin (JetHT) instance at 0x7f75d1deabe0>, <StringBin (QCD_HT1000to1500_TuneCP5_13TeV-madgraph-pythia8) instance at 0x7f75d1dea4f0>, <StringBin (QCD_HT1500to2000_TuneCP5_13TeV-madgraph-pythia8) ins

/opt/conda/lib/python3.8/site-packages/coffea/hist/hist_tools.py:376: RuntimeWarning: Not all requested indices present in <Cat (name=dataset) instance at 0x7f757d503a60>
  warnings.warn(


In [7]:
# Read the histogram from the pickle file

ggf = pickle.load(open(picklename,'rb')).integrate('region','signal-ggf')
vbf = pickle.load(open(picklename,'rb')).integrate('region','signal-vbf')
mucr = pickle.load(open(picklename,'rb')).integrate('region','muoncontrol')

In [8]:
vbf.identifiers('systematic')

[<StringBin (JERDown) instance at 0x7f75d1cde6d0>,
 <StringBin (JERUp) instance at 0x7f75d1cde670>,
 <StringBin (JESDown) instance at 0x7f75d1cde550>,
 <StringBin (JESUp) instance at 0x7f75d1cde4f0>,
 <StringBin (UESDown) instance at 0x7f75d1cde610>,
 <StringBin (UESUp) instance at 0x7f75d1cde5b0>,
 <StringBin (W_d2kappa_EWDown) instance at 0x7f75d1cde9d0>,
 <StringBin (W_d2kappa_EWUp) instance at 0x7f75d1cde730>,
 <StringBin (W_d3kappa_EWDown) instance at 0x7f75d1cdea90>,
 <StringBin (W_d3kappa_EWUp) instance at 0x7f75d1cdea30>,
 <StringBin (Z_d2kappa_EWDown) instance at 0x7f75d1cdec10>,
 <StringBin (Z_d2kappa_EWUp) instance at 0x7f75d1cdecd0>,
 <StringBin (Z_d3kappa_EWDown) instance at 0x7f75d1cdebb0>,
 <StringBin (Z_d3kappa_EWUp) instance at 0x7f75d1cdec70>,
 <StringBin (btagEffStatDown) instance at 0x7f75d1cde310>,
 <StringBin (btagEffStatUp) instance at 0x7f75d1cde250>,
 <StringBin (btagWeightDown) instance at 0x7f75d1cde490>,
 <StringBin (btagWeightUp) instance at 0x7f75d1cde3d0>

In [9]:
os.system('rm '+year+'/1-signalregion.root')
fout = uproot3.create(year+'/1-signalregion.root')
for p in pmap.keys():  
    if 'VBF_' in p or 'WH_' in p or 'ZH_' in p:
        continue
    
    print(p)
    if "data" in p:
        s = "nominal"
        h = vbf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
        fout["vbf_pass_"+p+"_"+s] = hist.export1d(h)
        h = vbf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
        fout["vbf_fail_"+p+"_"+s] = hist.export1d(h)
        
        h = ggf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
        fout["ggf_pass_"+p+"_"+s] = hist.export1d(h)
        h = ggf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
        fout["ggf_fail_"+p+"_"+s] = hist.export1d(h)

    else:
        for s in systematics:
            h = vbf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
            fout["vbf_pass_"+p+"_"+s] = hist.export1d(h)
            h = vbf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
            fout["vbf_fail_"+p+"_"+s] = hist.export1d(h)
            
            h = ggf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
            fout["ggf_pass_"+p+"_"+s] = hist.export1d(h)
            h = ggf.sum('pt1').integrate('systematic',s).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
            fout["ggf_fail_"+p+"_"+s] = hist.export1d(h)

fout.close()

ZH
WH
ttH
VBF
ggF
ggF-powheg
QCD
VV
Wjets
Zjets
ttbar
singlet
data
muondata


In [10]:
ptbins = [450, 550, 1200]
os.system('rm '+year+'/2pt-signalregion.root')
fout = uproot3.create(year+'/2pt-signalregion.root')

for i,b in enumerate(ptbins[:-1]):
    for p in pmap.keys(): 
        if 'VBF_' in p or 'WH_' in p or 'ZH_' in p:
            continue
        
        print(p)
        if "data" in p:
            s = "nominal"
            h = vbf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
            fout["vbf_pass_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
            h = vbf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
            fout["vbf_fail_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
            
            h = ggf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
            fout["ggf_pass_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
            h = ggf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
            fout["ggf_fail_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
            
        else:
            for s in systematics:
                h = vbf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
                fout["vbf_pass_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
                h = vbf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
                fout["vbf_fail_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
                
                h = ggf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
                fout["ggf_pass_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)
                h = ggf.integrate('systematic',s).integrate('pt1',int_range=slice(ptbins[i],ptbins[i+1])).integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
                fout["ggf_fail_pt"+str(i+1)+"_"+p+"_"+s] = hist.export1d(h)

fout.close()

ZH
WH
ttH
VBF
ggF
ggF-powheg
QCD
VV
Wjets
Zjets
ttbar
singlet
data
muondata
ZH
WH
ttH
VBF
ggF
ggF-powheg
QCD
VV
Wjets
Zjets
ttbar
singlet
data
muondata


In [11]:
os.system('rm '+year+'/muonCR.root')
fout = uproot3.create(year+'/muonCR.root')
for p in pmap.keys():  
    if 'VBF_' in p or 'WH_' in p or 'ZH_' in p:
         continue
            
    print(p)
    if "data" in p:
        s = "nominal"
        h = mucr.integrate('systematic',s).sum('pt1').integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
        fout["pass_"+p+"_"+s] = hist.export1d(h)
        h = mucr.integrate('systematic',s).sum('pt1').integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
        fout["fail_"+p+"_"+s] = hist.export1d(h)
    else:
        for s in systematics:
            h = mucr.integrate('systematic',s).sum('pt1').integrate('ddb1',int_range=slice(ddbthr,1)).integrate('process',p)
            fout["pass_"+p+"_"+s] = hist.export1d(h)
            h = mucr.integrate('systematic',s).sum('pt1').integrate('ddb1',int_range=slice(0,ddbthr)).integrate('process',p)
            fout["fail_"+p+"_"+s] = hist.export1d(h)

ZH
WH
ttH
VBF
ggF
ggF-powheg
QCD
VV
Wjets
Zjets
ttbar
singlet
data
muondata
